# Elevator Environment

In this Jupyter Notebook, we are going to learn how the elevator environment works.

## Setup

### Google Colab

If you are using Google Colab (and we encourage you to do so), please run the following code cell. If you are not using Google Colab, you can skip this code cell.

**Note**: If your assignment folder in Google Drive is located at `My Drive -> CSxx46-Assignment-1`, you should set `CODE_DIR` to `'MyDrive/CSxx46-Assignment-1'`.

In [ ]:
import os
import sys

DRIVE_DIR = '/content/drive/' # Google Colab mount point, do not change
CODE_DIR = 'MyDrive/CSxx46-Assignment-1' # Folder in your Google Drive where the code is stored

try:
    from google.colab import drive
    drive.mount(DRIVE_DIR)
    sys.path.append(os.path.join(DRIVE_DIR, CODE_DIR))
    # Update based on the Google Drive directory
    %cd /content/drive/MyDrive/CSxx46-Assignment-1
except:
    print("Not in Colab environment, skipping drive mount")
    pass

#### Installing Dependencies

The elevator task is implemented using the `PyRDDLGym` library. Before we begin, please install the following packages.

**Note**: You may need to restart the session. Please follow the prompt to do so.

In [ ]:
!pip install pyRDDLGym==2.6
!pip install rddlrepository==2.1

### Anaconda

If you prefer Anaconda/Miniconda, you can create the exact course environment from the provided `environment.yml`.

1. Install Anaconda (or Miniconda) and open an **Anaconda Prompt** (Windows) / terminal (macOS/Linux).
2. From the assignment folder, create the environment:
   ```bash
   conda env create -f environment.yml
   ```
3. Activate it:
   ```bash
   conda activate csxx46-ay2526s2-assignment-1
   ```
4. Launch JupyterLab (or VS Code) inside the env:
   ```bash
   jupyter lab
   ```

Useful links:
- Anaconda installation guide: https://docs.anaconda.com/anaconda/install/
- Anaconda Navigator getting started: https://docs.anaconda.com/navigator/

## Environment

The Elevator environment models evening rush hours when people from different floors in a building want to go down to the bottom floor using elevators.

The building has 5 floors and 2 elevator. Each floor may have several people waiting. The objective is to pick up passengers from various floors and deliver them to the first floor. New passengers may arrive at each floor while the elevator is in operation. The elevator can move up and down, and pick up and drop off passengers. However, it can only do so when the door is open, and it can only move when the door is closed.

In [ ]:
from gymnasium.wrappers import RecordEpisodeStatistics
from pyRDDLGym.core.env import RDDLEnv
from utils import DictToListWrapper

### Initialization

To initialize the environment, call the `Elevator` class. Here, we will use the `DictToListWrapper` to convert the environment's state from a dictionary to a list, with the detail given in the "Environment Description" section below.

In [ ]:
def create_elevator_env():
    env = RDDLEnv(
        domain="elevator/domain.rddl",
        instance="elevator/instance.rddl",  # instance-5 file
    )
    # If your observation is a Dict of booleans, flatten it:
    env = DictToListWrapper(env)
    env = RecordEpisodeStatistics(env)
    return env


env = create_elevator_env().env

### Observation and Action Space

Using the following code cell, we can check the observation and action space of the environment, with the detailed descriptions.

In [ ]:
print(f"Observation space: {env.observation_space}")
env.get_state_description()

print(f"Action space: {env.action_space}")
env.get_action_description()

### Interaction

The agent interacts with the environment following the [Gymnasium API](https://gymnasium.farama.org/). The environment provides the following methods:

- `reset()`: Resets the environment and returns the initial state along with any additional information (usually empty).
- `step(action)`: Takes an action in the environment and returns:
    - *next state*: The resulting state after the action.
    - *reward*: The reward received for the action.
    - *done*: A boolean indicating whether the episode has ended.
    - *truncated*: A boolean indicating whether the episode was truncated (terminated for any unspecified reason), though this is not applicable to our task.
    - *info*: Additional information returned as a dictionary.
- `close()`: Closes the environment and releases any resources.

A template for interacting with the environment is shown below:

In [ ]:
state, info = env.reset()

for i in range(20):
    # randomly sample an action from the action space
    action = env.action_space.sample()

    print(f"Action: {action}")

    next_state, reward, terminated, truncated, info = env.step(action)

    print(f"Next state: {next_state}")
    print(f"Reward: {reward}")
    print(f"Terminated: {terminated}")
    print(f"Truncated: {truncated}")

    done = terminated or truncated
    if done:
        state, info = env.reset()